# Which pages should an editor review first?

**Om Sawarkar · Capstone research assembly**

This notebook rebuilds the public paper from the saved executed Week 5–7 outputs. It does not rerun gated warehouse analysis. For a fresh model run, follow the linked analysis notebooks. No new sealed evaluation is claimed.

## 1. Question and abstract

Can a transparent CTR peer-gap baseline help prioritize editor review, and does logistic regression improve the ranking? The observed comparison supports keeping the baseline. The intended action is human investigation, not automatic editing. Update benefit remains unmeasured.

In [1]:
from pathlib import Path
import json, subprocess, sys

# Find the repository when local, or clone it in a fresh Colab runtime.
root = Path.cwd()
if not (root / "work/notebooks/w06_validation_audit.ipynb").exists():
    candidates = [p for p in [root, *root.parents] if (p / "work/notebooks/w06_validation_audit.ipynb").exists()]
    if candidates:
        root = candidates[0]
    else:
        destination = root / "capstone_repository"
        if not destination.exists():
            subprocess.run(["git", "clone", "https://github.com/omsawarkar/omsawarkar.git", str(destination)], check=True)
        root = destination
print("Repository located.")


Repository located.


## 2. Data

Release v20260703; March partition only: 9,841,378 daily rows. Inputs: 1–20 March, both availability flags TRUE, ≥10 observed days. Historical queue: 6,607 items. Evaluation: 6,531 eligible items from 21 clients with sufficient 21–31 March GSC outcomes. Dimensions and query table are not joined; private identifiers are omitted from the paper.

In [2]:
for name in ["w05_model", "w06_validation_audit", "w07_action_playbook"]:
    notebook = json.loads((root / "work/notebooks" / (name + ".ipynb")).read_text())
    errors = [o for c in notebook["cells"] for o in c.get("outputs", []) if o.get("output_type") == "error"]
    assert not errors, name + " contains an error output"
print("Saved analysis notebooks contain no recorded errors.")


Saved analysis notebooks contain no recorded errors.


## 3. Methodology

Baseline = impressions × max(peer-median CTR − item CTR, 0). Peer groups use position and volume, with at least 20 items. Logistic regression uses early impressions, clicks, CTR, sessions, and engagement rate; preprocessing is fitted within training folds. The outcome is later CTR below the frozen early peer benchmark. Client-grouped splitting prevents client overlap, while shared early reference benchmarks and outcome-based eligibility remain disclosed limitations. Random seed: 42.

## 4. Results versus baseline

Original split: 246 items, base rate 0.6911. Pooled grouped folds: 6,531 items, base rate 0.6567. Both methods tie at pooled Precision@10 and @20. Pooled baseline average precision / ROC-AUC: 0.8566 / 0.7840; model: 0.8025 / 0.6787. Fold imbalance and cross-model probability calibration limit pooled interpretation. The tables are extracted and checked below.

In [3]:
# Generate charts, extracted metric receipt, report and self-contained HTML.
try:
    import matplotlib
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "matplotlib"], check=True)
result = subprocess.run([sys.executable, str(root / "work/scripts/build_research_paper.py")], cwd=root, check=True, capture_output=True, text=True)
print(result.stdout)
metrics = json.loads((root / "work/outputs/paper_metrics.json").read_text())
for row in metrics["comparison"]:
    print(row["stage"], "|", row["method"], "| AP:", row["average_precision"], "| AUC:", row["roc_auc"])


Paper rebuilt from saved executed outputs.
Original comparison: 246 items; pooled comparison: 6,531 items.
Wrote docs/index.html, work/capstone_report.md, two figures and paper_metrics.json.

Before: original split | Week 4 baseline | AP: 0.8399 | AUC: 0.7321
Before: original split | Logistic regression | AP: 0.8648 | AUC: 0.7711
After: pooled out-of-fold | Week 4 baseline | AP: 0.8566 | AUC: 0.7840
After: pooled out-of-fold | Logistic regression | AP: 0.8025 | AUC: 0.6787


## 5. Limitations

This is an exploratory one-month observed-proxy analysis, not evidence of intervention benefit. Broad peers mix client/query differences. Outcome-coverage selection is retrospective. The large client in Fold 1 dominates pooled metrics; the original test set was already inspected. Human review precision, uncertainty intervals, and update benefit were not measured. June remains unused.

## 6. Ranked recommendations

Retain the transparent baseline. The historical queue has 3,280 review candidates, 3,285 monitoring items and 42 insufficient-data items. All remain NOT_REVIEWED. A proposed pilot allocates 15 minutes per item for ten items, with owner approval before changes. Cost targets and monitoring thresholds are assumptions rather than achieved results.

In [4]:
receipt = json.loads((root / "work/outputs/action_playbook_receipt.json").read_text())
assert receipt["queue_items"] == 6607
assert receipt["actual_page_reviews_completed"] == 0
print("Historical action counts:")
for action, count in receipt["action_counts"].items():
    print(action, count)


Historical action counts:
MONITOR 3285
REVIEW_SEARCH_APPEARANCE_AND_INTENT 3280
COLLECT_MORE_DATA 42


## 7. Artifacts, reproducibility and credit

The build script writes docs/index.html, work/capstone_report.md, work/outputs/paper_metrics.json, and two work/figures SVG charts. The HTML embeds charts, requiring no external assets. Existing receipts are preserved. Deploy docs through GitHub Pages, then record the verified live URL in submission/paper_url.txt.

Built on the [FlyRank ML Internship dataset](https://flyrank.ai). AI assistance supported code and report drafting. Om Sawarkar must review the paper before publication. Design references and source notebook links appear in the HTML.

In [5]:
page = (root / "docs/index.html").read_text()
for required in ["Abstract", "Methodology", "Limitations", "Reproducibility", "Acknowledgments", "https://flyrank.ai"]:
    assert required.lower() in page.lower(), required
assert page.count("<svg") == 2
assert "hf_" not in page
assert "client_" not in page
print("Required sections, data credit, embedded charts and public-safety checks passed.")
print("Deployment is a separate step: verify the live URL before recording it.")


Required sections, data credit, embedded charts and public-safety checks passed.
Deployment is a separate step: verify the live URL before recording it.


## Self-check

- [x] Report assembly cells executed from top to bottom without errors.
- [x] Figures and tables trace to saved executed analysis outputs.
- [x] The public paper omits private records and token values.
- [x] Proxy, selection and validation limitations are disclosed.
- [ ] Author has reviewed the final page.
- [ ] Files committed, live page checked, and exact URL recorded.
